<a href="https://colab.research.google.com/github/Detio1/Deep_Learning/blob/main/Hugging_Face_Fine_Tuning_Practice_Step_by_Step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



```
## Step 0 — Install libraries
```



In [2]:
!pip -q install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


## Step 1 — Import libraries

In [3]:
import random
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import evaluate

## Step 2 — Set seed and basic configuration

In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)


TASK_NAME = "Sentiment Classification"
MODEL_NAME = "distilbert-base-uncased"
DATASET_NAME = "glue"
DATASET_CONFIG = "sst2"


TRAIN_SAMPLES = 2000
VAL_SAMPLES = 500
MAX_LENGTH = 128
NUM_LABELS = 2
OUTPUT_DIR = "./distilbert-sst2-colab-notebook"


LABELS = {0: "NEGATIVE", 1: "POSITIVE"}
ID2LABEL = {0: "NEGATIVE", 1: "POSITIVE"}
LABEL2ID = {"NEGATIVE": 0, "POSITIVE": 1}


device = "cuda" if torch.cuda.is_available() else "cpu"


print("Task:", TASK_NAME)
print("Model:", MODEL_NAME)
print("Dataset:", f"{DATASET_NAME}/{DATASET_CONFIG}")
print("Device:", device)

Task: Sentiment Classification
Model: distilbert-base-uncased
Dataset: glue/sst2
Device: cuda


## Step 3 — Define the task in simple words

Before fine-tuning, clearly define what the model should learn.

Here the goal is: input: a sentence; output: **POSITIVE** or **NEGATIVE**

### Example

"This movie was amazing." → POSITIVE
"The service was terrible." → NEGATIVE

In [5]:
print("Goal: Train a model to classify sentiment.")
print("Example 1: 'This movie was amazing.' -> POSITIVE")
print("Example 2: 'The service was terrible.' -> NEGATIVE")

Goal: Train a model to classify sentiment.
Example 1: 'This movie was amazing.' -> POSITIVE
Example 2: 'The service was terrible.' -> NEGATIVE


## Step 4 — Load tokenizer and pretrained model

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print("Tokenizer loaded.")
print("Model loaded.")
print("Model type:", model.__class__.__name__)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer loaded.
Model loaded.
Model type: DistilBertForSequenceClassification


## Step 5 — Load the dataset

We use **GLUE / SST-2**, a sentiment analysis dataset.

It contains: training split; validation split; test split

In [7]:
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)

train_ds = dataset["train"].shuffle(seed=42).select(range(TRAIN_SAMPLES))
val_ds = dataset["validation"].shuffle(seed=42).select(range(VAL_SAMPLES))

print("Training samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Available columns:", train_ds.column_names)

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Training samples: 2000
Validation samples: 500
Available columns: ['sentence', 'label', 'idx']


## Step 6 — Inspect a few dataset examples

In [8]:
for i in range(3):
    row = train_ds[i]
    print(f"Example {i+1}")
    print("Sentence:", row["sentence"])
    print("Label   :", LABELS[row["label"]])
    print("-" * 60)

Example 1
Sentence: klein , charming in comedies like american pie and dead-on in election , 
Label   : POSITIVE
------------------------------------------------------------
Example 2
Sentence: be fruitful 
Label   : POSITIVE
------------------------------------------------------------
Example 3
Sentence: soulful and 
Label   : POSITIVE
------------------------------------------------------------


## Step 7 — Understand tokenization on one sentence

In [9]:
sample_text = train_ds[0]["sentence"]
tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Original sentence:")
print(sample_text)

print("\nTokens:")
print(tokens[:25])

print("\nToken IDs:")
print(token_ids[:25])

Original sentence:
klein , charming in comedies like american pie and dead-on in election , 

Tokens:
['klein', ',', 'charming', 'in', 'comedies', 'like', 'american', 'pie', 'and', 'dead', '-', 'on', 'in', 'election', ',']

Token IDs:
[12555, 1010, 11951, 1999, 22092, 2066, 2137, 11345, 1998, 2757, 1011, 2006, 1999, 2602, 1010]


## Step 8 — Tokenize the whole dataset

In [10]:
def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_train = train_ds.map(tokenize_function, batched=True)
tokenized_val = val_ds.map(tokenize_function, batched=True)
print("Tokenization complete.")
print("New columns:", tokenized_train.column_names)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization complete.
New columns: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']


## Step 9 — Inspect one tokenized example

In [11]:
example = tokenized_train[0]
print("Original text:")
print(train_ds[0]["sentence"])

print("\nLabel:")
print(LABELS[train_ds[0]["label"]])

print("\ninput_ids:")
print(example["input_ids"][:30])

print("\nattention_mask:")
print(example["attention_mask"][:30])

Original text:
klein , charming in comedies like american pie and dead-on in election , 

Label:
POSITIVE

input_ids:
[101, 12555, 1010, 11951, 1999, 22092, 2066, 2137, 11345, 1998, 2757, 1011, 2006, 1999, 2602, 1010, 102]

attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Step 10 — Prepare final training format

In [12]:
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_val = tokenized_val.rename_column("label", "labels")

keep_cols = ["input_ids", "attention_mask", "labels"]
tokenized_train = tokenized_train.remove_columns(
    [c for c in tokenized_train.column_names if c not in keep_cols]
)
tokenized_val = tokenized_val.remove_columns(
    [c for c in tokenized_val.column_names if c not in keep_cols]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Final training columns:", tokenized_train.column_names)
print("\nExample final row:")
print(tokenized_train[0])

Final training columns: ['labels', 'input_ids', 'attention_mask']

Example final row:
{'labels': 1, 'input_ids': [101, 12555, 1010, 11951, 1999, 22092, 2066, 2137, 11345, 1998, 2757, 1011, 2006, 1999, 2602, 1010, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## Step 11 — Define evaluation metrics

In [13]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")
    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
    }
print("Metrics ready: accuracy and f1")

Metrics ready: accuracy and f1


## Step 12 — Configure training settings

These parameters control how learning happens.

### Key settings

`learning_rate`: how fast the model updates
`batch_size`: number of samples per step
`num_train_epochs`: number of full passes over the data

In [14]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    fp16=torch.cuda.is_available(),
)
print("Training settings:")
print("learning_rate:", training_args.learning_rate)
print("train_batch_size:", training_args.per_device_train_batch_size)
print("eval_batch_size:", training_args.per_device_eval_batch_size)
print("num_train_epochs:", training_args.num_train_epochs)
print("eval_strategy:", training_args.eval_strategy)

Training settings:
learning_rate: 2e-05
train_batch_size: 16
eval_batch_size: 32
num_train_epochs: 1
eval_strategy: IntervalStrategy.EPOCH


## Step 13 — Create the Trainer

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Trainer created successfully.")

Trainer created successfully.


## Step 14 — Train the model

In [16]:
train_result = trainer.train()
print("Training finished.")
print("Training loss:", round(train_result.training_loss, 4))

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.420041,0.406386,0.822000,0.819473


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training finished.
Training loss: 0.51


## Step 15 — Evaluate the model

Now test the model on validation data that it did not train on.

Look at: `eval_loss`; `eval_accuracy`; `eval_f1`

In [17]:
metrics = trainer.evaluate()
metrics

{'eval_loss': 0.4063856303691864,
 'eval_accuracy': 0.822,
 'eval_f1': 0.8194726166328601,
 'eval_runtime': 0.3127,
 'eval_samples_per_second': 1598.745,
 'eval_steps_per_second': 51.16,
 'epoch': 1.0}

## Step 16 — Read the evaluation outcome

In [18]:
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

eval_loss: 0.4064
eval_accuracy: 0.8220
eval_f1: 0.8195
eval_runtime: 0.3127
eval_samples_per_second: 1598.7450
eval_steps_per_second: 51.1600
epoch: 1.0000


## Step 17 — Save the fine-tuned model

In [19]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to:", OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: ./distilbert-sst2-colab-notebook


## Step 18 — Test the fine-tuned model on new sentences

In [20]:
test_sentences = [
    "This product is fantastic and works perfectly.",
    "I am very disappointed with the quality.",
    "The experience was okay, not the best but acceptable.",
]
inputs = tokenizer(test_sentences, padding=True, truncation=True, return_tensors="pt").to(model.device)
model.eval()
with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = np.argmax(probs, axis=-1)

for text, pred, prob in zip(test_sentences, preds, probs):
    print("Text:", text)
    print("Predicted label:", LABELS[int(pred)])
    print(f"Confidence -> NEGATIVE: {prob[0]:.4f}, POSITIVE: {prob[1]:.4f}")
    print("-" * 70)

Text: This product is fantastic and works perfectly.
Predicted label: POSITIVE
Confidence -> NEGATIVE: 0.0734, POSITIVE: 0.9266
----------------------------------------------------------------------
Text: I am very disappointed with the quality.
Predicted label: NEGATIVE
Confidence -> NEGATIVE: 0.6785, POSITIVE: 0.3215
----------------------------------------------------------------------
Text: The experience was okay, not the best but acceptable.
Predicted label: POSITIVE
Confidence -> NEGATIVE: 0.2173, POSITIVE: 0.7827
----------------------------------------------------------------------
